# 🧭 Tree Traversal — Runnable Notebook

Companion to [`README.md`](README.md) and
[`04_tree_traversal_orders_lesson.html`](04_tree_traversal_orders_lesson.html).

Four orders on one tree: **pre / in / post** (DFS) and **level** (BFS).

## 1. The tree and the three recursive DFS orders
The only difference is *when* you visit the node relative to its children.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val = val; self.left = left; self.right = right

#          1
#         / \
#        2   3
#       / \ / \
#      4  5 6  7
root = TreeNode(1,
    TreeNode(2, TreeNode(4), TreeNode(5)),
    TreeNode(3, TreeNode(6), TreeNode(7)))

def preorder(n, out=None):
    """Node, Left, Right."""
    out = [] if out is None else out
    if n:
        out.append(n.val); preorder(n.left, out); preorder(n.right, out)
    return out

def inorder(n, out=None):
    """Left, Node, Right."""
    out = [] if out is None else out
    if n:
        inorder(n.left, out); out.append(n.val); inorder(n.right, out)
    return out

def postorder(n, out=None):
    """Left, Right, Node."""
    out = [] if out is None else out
    if n:
        postorder(n.left, out); postorder(n.right, out); out.append(n.val)
    return out

print("pre :", preorder(root))     # root first
print("in  :", inorder(root))      # sorted, if this were a BST
print("post:", postorder(root))    # root last
assert preorder(root)  == [1, 2, 4, 5, 3, 6, 7]
assert inorder(root)   == [4, 2, 5, 1, 6, 3, 7]
assert postorder(root) == [4, 5, 2, 6, 7, 3, 1]

## 2. Iterative DFS (an explicit stack)
When recursion is risky (very deep trees), make the stack yourself.

In [ ]:
def preorder_iter(root):
    """Push RIGHT before LEFT, so LEFT pops first (Node, Left, Right)."""
    if not root:
        return []
    out, stack = [], [root]
    while stack:
        node = stack.pop()            # LIFO -> depth-first
        out.append(node.val)
        if node.right: stack.append(node.right)
        if node.left:  stack.append(node.left)
    return out

def inorder_iter(root):
    """Walk left pushing nodes; pop to visit; then go right."""
    out, stack, node = [], [], root
    while stack or node:
        while node:                   # dive as far LEFT as possible
            stack.append(node)
            node = node.left
        node = stack.pop()            # deepest unvisited node
        out.append(node.val)
        node = node.right             # now handle its right subtree
    return out

assert preorder_iter(root) == preorder(root)
assert inorder_iter(root)  == inorder(root)
print("iterative pre:", preorder_iter(root))
print("iterative in :", inorder_iter(root))

## 3. Level-order (BFS) grouped by level
The `for _ in range(len(q))` snapshot processes exactly one row per outer loop.

In [ ]:
from collections import deque

def level_order(root):
    """BFS grouped by level: each output entry is one full row."""
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):       # freeze this level's size ...
            node = q.popleft()
            level.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        out.append(level)             # ... so each entry is one level
    return out

print("levels:", level_order(root))
assert level_order(root) == [[1], [2, 3], [4, 5, 6, 7]]

## 4. Rebuild a tree from two traversals
**pre-order + in-order** uniquely determine a tree: pre[0] is the root; its spot in in-order splits left | right.

In [ ]:
def build(preorder, inorder):
    """Rebuild the unique tree from pre-order + in-order lists."""
    if not preorder:
        return None
    root_val = preorder[0]            # pre-order's first element is the root
    root = TreeNode(root_val)
    mid = inorder.index(root_val)     # its position splits in-order into left | right
    root.left  = build(preorder[1:mid + 1], inorder[:mid])
    root.right = build(preorder[mid + 1:],  inorder[mid + 1:])
    return root

rebuilt = build([1, 2, 4, 5, 3, 6, 7], [4, 2, 5, 1, 6, 3, 7])
print("rebuilt level-order:", level_order(rebuilt))
assert level_order(rebuilt) == level_order(root)   # same tree recovered

## ✅ Recap
| Order | Use it for |
|---|---|
| **Pre** (Node,L,R) | copy / serialize (top-down) |
| **In** (L,Node,R) | **BST sorted output** |
| **Post** (L,R,Node) | height / size / delete (bottom-up) |
| **Level** (BFS) | shortest path, per-level work |

- DFS = **stack/recursion**; BFS = **queue**.
- **in-order + (pre or post)** rebuilds a unique tree.

Next: the graph track — [`05_Graph_Fundamentals`](../05_Graph_Fundamentals/README.md).